# ContractRisk M3: desarrollo v2

Comparación de recuperación: denso, híbrido con filtro de ID, e híbrido con reranking. Mismo prompt v2 y generador en todas las variantes. Cuatro preguntas de desarrollo y un caso adicional de contexto vacío, sin gold final. No es aún el harness final, el agente o RAGAS. Usa Colab T4 y un entorno nuevo. Sube contractrisk_m3_desarrollo_v2_codigo.zip. La solución SQLite es automática; no copies las celdas de reparación anteriores.

In [1]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess
uploaded = files.upload()
name = 'contractrisk_m3_desarrollo_v2_codigo.zip'
assert name in uploaded, f'Sube {name}'
root = Path('/content/contractrisk_m3_desarrollo_v2')
assert not root.exists(), 'Usa un entorno nuevo; no mezcles corridas.'
with zipfile.ZipFile(name) as z:
    for member in z.namelist():
        assert (root / member).resolve().is_relative_to(root.resolve()), 'Ruta insegura'
    z.extractall(root)
os.chdir(root)
print(root)

Saving contractrisk_m3_desarrollo_v2_codigo.zip to contractrisk_m3_desarrollo_v2_codigo.zip
/content/contractrisk_m3_desarrollo_v2


In [2]:
subprocess.run([sys.executable,'-m','pip','install','-r','m3/requirements-development-v2.txt'],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],check=True)
subprocess.run([sys.executable,'-c',"import sys;sys.path.insert(0,'m3');from sqlite_support import ensure_sqlite;print(ensure_sqlite());import torch;assert torch.cuda.is_available(),'Activa T4';print(torch.cuda.get_device_name(0))"],check=True)

CompletedProcess(args=['/usr/bin/python3', '-c', "import sys;sys.path.insert(0,'m3');from sqlite_support import ensure_sqlite;print(ensure_sqlite());import torch;assert torch.cuda.is_available(),'Activa T4';print(torch.cuda.get_device_name(0))"], returncode=0)

In [3]:
command = [sys.executable,'-u','m3/run_development_v2.py','--output','desarrollo_v2_resultados']
with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:
        print(line,end='',flush=True)
    returncode = process.wait()
if returncode:
    print('La corrida falló. Ejecuta la siguiente celda para descargar el diagnóstico.')
else:
    print('Corrida terminada. Descarga los resultados para revisar calidad.')

I0000 00:00:1793243421.642112    4280 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793243424.165232    4280 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Building dense index with contract identity...
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Loading reranker...

Fetching 7 files: 100%|██████████| 7/7 [00:10<00:00,  1.5

In [4]:
import shutil
resultados = Path('desarrollo_v2_resultados')
assert resultados.exists(), 'Si falló antes de crear resultados, envía el error completo de instalación/preflight.'
print((resultados/'metadata.json').read_text())
archive = shutil.make_archive('/content/contractrisk_m3_desarrollo_v2_resultados','zip',resultados)
files.download(archive)

{
  "status": "completed",
  "purpose": "development_v2_not_final_scorecard",
  "started_utc": "2026-10-29T03:10:11.594159+00:00",
  "python": "3.13.16",
  "seed": 42,
  "sqlite": {
    "backend": "sqlite3",
    "sqlite_version": "3.45.1",
    "fts5_checked": true
  },
  "gpu": "Tesla T4",
  "packages": {
    "torch": "2.11.0+cu130",
    "numpy": "2.1.3",
    "transformers": "4.57.6",
    "sentence-transformers": "3.4.1",
    "chromadb": "0.6.3",
    "rank-bm25": "0.2.2",
    "pysqlite3-binary": "0.5.4.post2"
  },
  "models": {
    "embedding": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "generator": "Qwen/Qwen2.5-1.5B-Instruct",
    "reranker": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
  },
  "revisions": {
    "embedding": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
    "generator": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    "reranker": "1427fd652930e4ba29e8149678df786c240d8825"
  },
  "corpus_sha256": "7c179b53b87103e6b2992ffda49775bce6118f1ab577c991c4

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Devuelve el ZIP y este notebook con sus salidas. Conserva la corrida anterior. No ajustes los prompts ni los datos para subir puntuaciones. Los 12 casos gold no se ejecutan en esta prueba.